In [1]:
from src.duck_db_utils import get_table_data,get_db_table_names, get_conn,close_conn

In [2]:
import importlib
import src.duck_db_utils as duck_db_utils

importlib.reload(duck_db_utils)
duck_db_utils.close_conn()
conn = duck_db_utils.get_conn()

# Keep helpers available with current module state
get_conn = duck_db_utils.get_conn
close_conn = duck_db_utils.close_conn
get_table_data = duck_db_utils.get_table_data
get_db_table_names = duck_db_utils.get_db_table_names

In [3]:
db_info = conn.execute("PRAGMA database_list").fetch_df()
active_db_path = db_info.loc[db_info["name"] == "stocks", "file"].iloc[0]
print(f"Using DB: {active_db_path}")
get_db_table_names(conn)

Using DB: c:\users\rocke\onedrive\1. daily_deliverables\1. master_folder\investment_strategy\project_automation\stock_db\src\data\stocks.duckdb


['refresh_log', 'stock_data', 'stocks', 'tbl_priority', 'todays_data']

In [4]:
stock_data = get_table_data(conn, "stock_data")
stock_data.trade_date.max()

Timestamp('2026-08-14 00:00:00')

In [5]:
stock_data.head(2)

,stock_name,stock_id,exchange,symbol,trade_date,open,high,low,close,adj_close,volume,created_at
0,TCPL Packaging Limited,1,NSE,TCPLPACK,2026-01-01,3015.899902,3059.000000,3002.100098,3041.399902,3018.010254,650,2026-08-16 21:47:42.086
1,TCPL Packaging Limited,1,NSE,TCPLPACK,2026-01-02,3036.800049,3052.800049,2999.500000,3006.000000,2982.882568,1687,2026-08-16 21:47:42.092


In [6]:
stock_data[stock_data['symbol']=='RELIANCE'].sort_values(by='trade_date', ascending=False).head(2)

,stock_name,stock_id,exchange,symbol,trade_date,open,high,low,close,adj_close,volume,created_at
24774,Reliance Industries Limited,9,NSE,RELIANCE,2026-08-14,1317.000000,1317.500000,1301.500000,1310.0,1310.0,10497358,2026-08-16 21:47:33.114
24773,Reliance Industries Limited,9,NSE,RELIANCE,2026-08-13,1327.800049,1327.800049,1307.199951,1317.0,1317.0,9397301,2026-08-16 21:47:33.109


In [7]:
import importlib
import src.priority_mapping as priority_mapping

importlib.reload(priority_mapping)

refresh_summary = priority_mapping.refresh_priority_for_all_symbols(conn)
tbl_priority = get_table_data(conn, "tbl_priority").sort_values(
    "current_yr_return", ascending=False
).reset_index(drop=True)

refresh_summary, tbl_priority.head(20)

({'source_table': 'stock_data',
  'target_table': 'tbl_priority',
  'processed_symbols': 13,
  'updated_rows': 13,
  'failed_symbols': []},
     stock_id                                 stock_name      symbol  \
 0         10           Lloyds Engineering Works Limited  LLOYDSENGG   
 1         13                          TAAL Tech Limited    TAALTECH   
 2          1                     TCPL Packaging Limited    TCPLPACK   
 3         12         Dixon Technologies (India) Limited       DIXON   
 4          6              Hindustan Aeronautics Limited         HAL   
 5         15  Kovai Medical Center and Hospital Limited      523323   
 6          8          Mazagon Dock Shipbuilders Limited     MAZDOCK   
 7          7                              Trent Limited       TRENT   
 8          5             The Supreme Industries Limited  SUPREMEIND   
 9          2                       Abbott India Limited  ABBOTINDIA   
 10        11               Sapphire Foods India Limited    SAPPHIRE

In [8]:
tbl_priority.to_csv("tbl_priority.csv", index=False)
stock_data.to_csv("stock_data.csv", index=False)